In [1]:
import torch

# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 3 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

using device: cpu


In [2]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [3]:
from dotenv import load_dotenv     # load HF token
load_dotenv()
from PIL import Image

## Image example
### Compute the masks

In [4]:
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_sam3_image_model

# Load the model
model = build_sam3_image_model()
processor = Sam3Processor(model)

/home/gllrm/workspace/fish-segmentation/sam3_src/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/gllrm/workspace/fish-segmentation/sam3_src/sam3/model/sam3_video_inference.py:800: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/home/gllrm/workspace/fish-segmentation/sam3_src/sam3/model/sam3_video_inference.py:908: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/home/gllrm/workspace/fish-segmentation/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.laye

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
# Load an image
image_path = "test_media/dogs.jpg"
image = Image.open(image_path)
image

In [ ]:
# inference + text prompt
IMAGE_TEXT_PROMPT = "dogs"

with torch.autocast(device, dtype=torch.bfloat16):
    inference_state = processor.set_image(image)
    output = processor.set_text_prompt(state=inference_state,
                                       prompt=IMAGE_TEXT_PROMPT)

In [ ]:
# Get the masks, bounding boxes, and scores
masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

### Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

masks_np = masks.cpu().float().numpy()
boxes_np = boxes.cpu().float().numpy()
scores_np = scores.cpu().float().numpy()

plt.figure(figsize=(10, 10))
plt.imshow(image)
ax = plt.gca()

for i in range(len(masks_np)):
    # mask
    mask = masks_np[i]
    if len(mask.shape) == 3: 
        mask = mask[0] # Flatten from (1, H, W) to (H, W) if needed
        
    # random color + transparency
    color = np.concatenate([np.random.random(3), np.array([0.5])], axis=0)
    h, w = mask.shape
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
    # box
    box = boxes_np[i]
    x_min, y_min, x_max, y_max = box
    rect = patches.Rectangle(
        (x_min, y_min), 
        x_max - x_min, 
        y_max - y_min, 
        linewidth=2, 
        edgecolor='red', 
        facecolor='none'
    )
    ax.add_patch(rect)
    
    # score
    score = scores_np[i]
    ax.text(
        x_min, y_min - 5, 
        f"Score: {score:.2f}", 
        color='white', 
        fontsize=10, 
        weight='bold', 
        bbox=dict(facecolor='red', edgecolor='red', alpha=0.8, pad=1)
    )

plt.axis('off')
plt.tight_layout()
plt.show()

## Video example

In [ ]:
from sam3.model_builder import build_sam3_multiplex_video_predictor
predictor = build_sam3_multiplex_video_predictor()

In [ ]:
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
)

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12


def propagate_in_video(predictor, session_id):
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]

    return outputs_per_frame


def abs_to_rel_coords(coords, IMG_WIDTH, IMG_HEIGHT, coord_type="point"):
    """Convert absolute coordinates to relative coordinates (0-1 range)

    Args:
        coords: List of coordinates
        coord_type: 'point' for [x, y] or 'box' for [x, y, w, h]
    """
    if coord_type == "point":
        return [[x / IMG_WIDTH, y / IMG_HEIGHT] for x, y in coords]
    elif coord_type == "box":
        return [
            [x / IMG_WIDTH, y / IMG_HEIGHT, w / IMG_WIDTH, h / IMG_HEIGHT]
            for x, y, w, h in coords
        ]
    else:
        raise ValueError(f"Unknown coord_type: {coord_type}")

In [ ]:
# "video_path" needs to be either a JPEG folder or a MP4 video file
video_path = "test_media/dogs_playing.mp4"

In [ ]:
# load "video_frames_for_vis" for visualization purposes (they are not used by the model)
if isinstance(video_path, str) and video_path.endswith(".mp4"):
    cap = cv2.VideoCapture(video_path)
    video_frames_for_vis = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        video_frames_for_vis.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
else:
    video_frames_for_vis = glob.glob(os.path.join(video_path, "*.jpg"))
    try:
        video_frames_for_vis.sort(
            key=lambda p: int(os.path.splitext(os.path.basename(p))[0])
        )
    except ValueError:
        print(
            f'frame names are not in "<frame_index>.jpg" format: {video_frames_for_vis[:5]=}, '
            f"falling back to lexicographic sort."
        )
        video_frames_for_vis.sort()

In [ ]:
# Save the original method
original_init_state = predictor.model.init_state

# Define a wrapper that removes the bad argument
def patched_init_state(*args, **kwargs):
    kwargs.pop('offload_state_to_cpu', None) # Remove the arg if it exists
    return original_init_state(*args, **kwargs)

# Overwrite the method with our patched version
predictor.model.init_state = patched_init_state

In [ ]:
# Now run your request as normal
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=video_path,
    )
)
session_id = response["session_id"]

In [ ]:
# note: in case you already ran one text prompt and now want to switch to another text prompt
# it's required to reset the session first (otherwise the results would be wrong)
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

In [ ]:
from torch.nn.attention import SDPBackend, sdpa_kernel


prompt_text_str = "dog"
frame_idx = 0  # add a text prompt on frame 0
response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=frame_idx,
        text=prompt_text_str,
    )
)

out = response["outputs"]

plt.close("all")

visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM 3.1 Dense Tracking outputs"],
    figsize=(6, 4),
)

In [ ]:
import gc
import torch

# Delete old large variables if you have any laying around
# del old_large_tensor 

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# now we propagate the outputs from frame 0 to the end of the video and collect all outputs
outputs_per_frame = propagate_in_video(predictor, session_id)

# finally, we reformat the outputs for visualization and plot the outputs every 60 frames
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

vis_frame_stride = 60
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM 3.1 Dense Tracking outputs"],
        figsize=(6, 4),
    )